In [1]:
!python -m mujoco.viewer --mjcf /home/robot/rl/FlapSim/flapsim/robot/SingleThrustFlapper/urdf/SingleThrustFlapper.urdf

In [23]:
import time
import numpy as np
import matplotlib.pyplot as plt
import mujoco
import mujoco.viewer

# 1. 加载 URDF

In [24]:

urdf_path = "/home/robot/rl/FlapSim/flapsim/robot/SingleThrustFlapper/urdf/SingleThrustFlapper.urdf"

model = mujoco.MjModel.from_xml_path(urdf_path)

# base_link 是固定基座，直接放到 z = 0.5 m
model.body("world").pos[:] = [0.0, 0.0, 0.5]

data = mujoco.MjData(model)
mujoco.mj_forward(model, data)

ValueError: Error: Error opening file '/home/robot/rl/FlapSim/flapsim/robot/SingleThrustFlapper/urdf/base_link.STL': No such file or directory

# 2. 关节名称与正弦运动幅值

In [ ]:
joint_names = [
    "pitch_joint",
    "roll_joint",
    "left_flapping_joint",
    "left_wing_feathering",
    "right_wing_flapping",
    "right_wing_feathering",
]

amplitudes = np.array([
    0.25,   # pitch_joint
    0.25,   # roll_joint
    1.20,   # left_flapping_joint
    0.60,   # left_wing_feathering
    1.20,   # right_wing_flapping
    0.60,   # right_wing_feathering
])

frequency = 0.5     # Hz
duration = 10.0     # 仿真持续时间
dt = 0.01           # 刷新间隔

# 3. 正弦驱动并记录轨迹

In [ ]:
time_history = []
q_history = []

with mujoco.viewer.launch_passive(model, data) as viewer:

    start_time = time.time()

    while viewer.is_running():

        t = time.time() - start_time

        if t > duration:
            break

        # 正弦关节位置
        q = amplitudes * np.sin(2.0 * np.pi * frequency * t)

        with viewer.lock():

            for name, q_value in zip(joint_names, q):
                data.joint(name).qpos[0] = q_value

            # 更新机器人各个 link 的位置
            mujoco.mj_forward(model, data)

        viewer.sync()

        time_history.append(t)
        q_history.append(q.copy())

        time.sleep(dt)

# 4. 绘制关节运动轨迹

In [ ]:
time_history = np.array(time_history)
q_history = np.array(q_history)

plt.figure(figsize=(10, 6))

for i, name in enumerate(joint_names):
    plt.plot(
        time_history,
        q_history[:, i],
        label=name
    )